# Run-to-run spread of every condition, by model and question

FA elicitation samples from the model, so every FA score is one draw. The
manuscript's stability appendix reports how far the same condition moves between
repeated runs, pooled by question. This notebook writes the granular table behind
that reading: one row per condition and question with the standard deviation of
each score across that condition's FA runs and the half-width of the 95%
confidence interval of its mean over those runs, $t_{0.975, k-1}\, s / \sqrt{k}$,
then rows that pool those deviations by degrees of freedom, $\sqrt{\sum_i (k_i - 1)\, s_i^2 / \sum_i (k_i - 1)}$, per
condition over the questions, per model, per variant and question over the models,
per variant, per question over everything, and over everything at once.

Only conditions run more than once enter: the six locally run models as released
(two runs) and under the German and Mexican-Spanish variants (three runs, two for
Muse-Glimmer-30B). GPT-5.6-Terra and the Mixtral references ran once and carry no
spread. NTP is deterministic under the decoding configuration and is left out.

Reads `outputs/culture/population_fidelity_groups.csv`, the table the fidelity
notebook wrote over every replicate, and writes
`outputs/culture/population_fidelity_replicates.csv`.


In [1]:
import pandas as pd

from culture.fidelity import GROUPS_TABLE
from culture.fidelity_replicates import (
    EVERY,
    REPLICATES_TABLE,
    SPREAD_SCORES,
    interval_column,
    pair_rows,
    replicate_table,
)
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv


## Build


In [2]:
groups = read_csv(GROUPS_TABLE)
assert groups is not None
table = replicate_table(groups)
write_csv(table, REPLICATES_TABLE)
pairs = pair_rows(table)
print(len(pairs), "condition-question pairs with runs per pair", pairs["n_runs"].value_counts().sort_index().to_dict(), "and", len(table) - len(pairs), "pooled rows")


72 condition-question pairs with runs per pair {3: 72} and 44 pooled rows


## Reading the table

Every row carries `n_pairs`, the condition-question pairs it pools, `n_runs`, the
runs behind them, and `degrees_of_freedom`, the sum of `n_runs - 1` over the pairs,
which is the weight each pair's variance gets. A row with `n_pairs == 1` is one
condition on one question, its `<score>_sd` is the plain standard deviation
across that condition's runs, and its `<score>_ci` is the half-width of the 95%
confidence interval of the mean over those runs, $t_{0.975,\,\mathrm{df}}\, s / \sqrt{k}$.
A pooled row takes the same half-width from its pooled deviation, its summed
degrees of freedom and its mean number of runs per pair, so a two-run pair reads
with the 12.7 quantile of one degree of freedom and the pooled rows with a
quantile near 2. The pooled rows spell `all` in every identity column
they pool over: `question == "all"` pools a condition's four questions, `arm ==
"all"` pools a model's variants, `model_label == "all"` pools the models.

The rows that spell `all` for the model and the variant reproduce the per-question
rows of the manuscript's stability table, and the row that spells `all` everywhere
its "All questions" row.


In [3]:
spread = [f"{score}_sd" for score in SPREAD_SCORES]
interval = [interval_column(score) for score in SPREAD_SCORES]
pooled = table[table["model_label"].eq(EVERY) & table["arm"].eq(EVERY)].set_index("question_label")
print(pooled[["n_pairs", "degrees_of_freedom", *spread, *interval]].round(3).to_string())


                n_pairs  degrees_of_freedom  score_accuracy_sd  score_dispersion_sd  score_structure_sd  pfs_sd  score_center_sd  score_accuracy_ci  score_dispersion_ci  score_structure_ci  pfs_ci  score_center_ci
question_label                                                                                                                                                                                                       
Happiness            18                  36              0.001                0.011               0.013   0.016            0.001              0.001                0.013               0.015   0.019            0.001
Politics             18                  36              0.001                0.007               0.015   0.023            0.001              0.001                0.008               0.017   0.027            0.001
Religion             18                  36              0.002                0.007               0.006   0.024            0.002              0.

## By model and question

The manuscript's granular table: one block of rows per model, one row per question
and an `all` row pooling the four, and one column group per variant with the five
scores as confidence half-widths. The closing block pools the models, per variant.


In [4]:
arms = ["base", "german", "spanish-mx"]
columns = pd.MultiIndex.from_product([arms, interval])
by_condition = table[table["model_label"].ne(EVERY) & table["arm"].ne(EVERY)]
wide = by_condition.pivot(index=["model_label", "question_label"], columns="arm", values=interval)
print(wide.swaplevel(axis=1).reindex(columns=columns).round(3).to_string())
by_arm = table[table["model_label"].eq(EVERY) & table["arm"].ne(EVERY)]
closing = by_arm.pivot(index="question_label", columns="arm", values=interval)
print(closing.swaplevel(axis=1).reindex(columns=columns).round(3).to_string())


                                                 base                                                                          german                                                                      spanish-mx                                                              
                                    score_accuracy_ci score_dispersion_ci score_structure_ci pfs_ci score_center_ci score_accuracy_ci score_dispersion_ci score_structure_ci pfs_ci score_center_ci score_accuracy_ci score_dispersion_ci score_structure_ci pfs_ci score_center_ci
model_label          question_label                                                                                                                                                                                                                                                
Gemma-4-31B-it       Happiness                  0.000               0.018              0.005  0.023           0.001             0.000               0.025              0.003

## Reading the spread

The pooled standard deviation of PFS is a few hundredths on every model, so the
half-width of a pooled row is a few hundredths too, while a single two-run pair
carries the 12.7 quantile of one degree of freedom and reads wide. The movement concentrates in a handful of pairs whose structure correlation crosses the
clip at zero, where the geometric mean sends the whole score to or from exactly
zero. Accuracy and center alignment, both read off individual cells, move by a
thousandth or two on every model and variant. The as-released conditions, with two
runs, are read on one degree of freedom per pair and pool less finely than the
finetuned ones.
